# IMPORTING LIB'S AND DEVICE

In [20]:
import os, json, math, random, glob, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torchvision as tv
from torchvision import transforms

from sklearn.model_selection import train_test_split

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [22]:
!nvidia-smi

Thu Sep 18 21:18:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.153.02             Driver Version: 570.153.02     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   61C    P0             16W /   35W |    1525MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# LOAD PATH

In [23]:
TRAIN_IMG_DIR = Path("../Datasets/train/images/")
TRAIN_LBL_DIR = Path("../Datasets/train/labels/")
TEST_IMG_DIR  = Path("../Datasets/test/images/")
SUB_TEMPLATE  = Path("../Datasets/sample_submission.csv")

# REPRODUCIBILITY

In [24]:
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# TRAINING HYPERPAR

In [25]:
CFG = dict(
    img_short_side=448,          # keep detail; resize by short-side, preserve aspect
    pad_to=32,                   # pad to multiple of 32 for CNN efficiency
    batch_size=2,                # tune to your GPU memory (RTX 3050: 4–8 is typical at 768p)
    num_workers=2,
    epochs=15,
    lr=0.005,
    weight_decay=1e-4,
    model_name="resnet50",       # 'resnet34' if VRAM is tight
    dropout=0.2,
    warmup_epochs=2,
    val_split=0.1,
    amp=True,                    # mixed precision
)

# HIGH FIDEL TRANSFORM

turned of the jitter

In [26]:
class LetterboxPad:
    """Pad to (H, W) that are multiples of 'pad_to', keeping content centered."""
    def __init__(self, pad_to=32, fill=0):
        self.pad_to = pad_to; self.fill = fill
    def __call__(self, img: Image.Image):
        w, h = img.size
        new_w = math.ceil(w / self.pad_to) * self.pad_to
        new_h = math.ceil(h / self.pad_to) * self.pad_to
        if new_w == w and new_h == h:
            return img
        out = Image.new(img.mode, (new_w, new_h), color=self.fill)
        out.paste(img, ((new_w - w)//2, (new_h - h)//2))
        return out

def build_transforms(train=True):
    tfms = []
    # 1) resize by short side -> preserves aspect
    tfms.append(transforms.Lambda(
        lambda im: tv.transforms.functional.resize(
            im, size=CFG["img_short_side"], max_size=None, antialias=True)))
    # 2) (optional) gentle photometric augmentation for train
    if train:
        tfms += [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomAdjustSharpness(sharpness_factor=1.3, p=0.3),
            transforms.RandomAutocontrast(p=0.2),
            transforms.Lambda(lambda im: tv.transforms.functional.adjust_gamma(im, gamma=0.95)),  # light gamma
    ]
    # 3) pad to multiple of 32 (letterbox)
    tfms.append(LetterboxPad(CFG["pad_to"]))
    # 4) to tensor + ImageNet norm
    tfms += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ]
    return transforms.Compose(tfms)


In [27]:
def _pad_to(img: torch.Tensor, H: int, W: int) -> torch.Tensor:
    """
    img: [C,H0,W0] -> pad bottom/right to [C,H,W]
    """
    _, h, w = img.shape
    pad_h, pad_w = H - h, W - w
    if pad_h == 0 and pad_w == 0:
        return img
    return F.pad(img, (0, pad_w, 0, pad_h), value=0)

def collate_pad_train(batch):
    imgs, ys = zip(*batch)
    H = max(t.shape[1] for t in imgs)
    W = max(t.shape[2] for t in imgs)
    imgs = torch.stack([_pad_to(t, H, W) for t in imgs], dim=0).to(memory_format=torch.channels_last)
    ys = torch.stack(ys, dim=0)
    return imgs, ys

def collate_pad_test(batch):
    imgs, names = zip(*batch)
    H = max(t.shape[1] for t in imgs)
    W = max(t.shape[2] for t in imgs)
    imgs = torch.stack([_pad_to(t, H, W) for t in imgs], dim=0).to(memory_format=torch.channels_last)
    return imgs, list(names)


# DATASET CLASS

In [28]:
class CrowdCountDataset(Dataset):
    def __init__(self, img_paths, lbl_dir: Path | None, train=True):
        self.img_paths = img_paths
        self.lbl_dir = lbl_dir
        self.train = train
        self.tfms = build_transforms(train=train)

    def _load_count(self, img_path: Path):
        if self.lbl_dir is None:
            return None
        stem = img_path.stem
        lbl_fp = self.lbl_dir / f"{stem}.json"
        with open(lbl_fp, "r") as f:
            js = json.load(f)
        # Expect 'human_num' in your JSON
        return float(js.get("human_num", 0.0))

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, i):
        img_path = self.img_paths[i]
        with Image.open(img_path) as im:
            im = im.convert("RGB")
        y = self._load_count(img_path)
        im = self.tfms(im)
        if y is None:
            return im, img_path.name  # test-time: return filename
        return im, torch.tensor([y], dtype=torch.float32)

In [29]:
train_imgs = sorted([Path(p) for p in glob.glob(str(TRAIN_IMG_DIR / "*")) if p.lower().endswith((".jpg",".jpeg",".png"))])
test_imgs = sorted([Path(p) for p in glob.glob(str(TEST_IMG_DIR / "*")) if p.lower().endswith((".jpg",".jpeg",".png"))])
assert len(train_imgs) >= 50, f"Found too few train images: {len(train_imgs)}"

tr_imgs, val_imgs = train_test_split(train_imgs, test_size=CFG["val_split"], random_state=42)
ds_tr = CrowdCountDataset(tr_imgs, TRAIN_LBL_DIR, train=True)
ds_va = CrowdCountDataset(val_imgs, TRAIN_LBL_DIR, train=False)
ds_te = CrowdCountDataset(test_imgs, lbl_dir=None, train=False)

dl_tr = DataLoader(ds_tr, batch_size=CFG["batch_size"], shuffle=True,
                   num_workers=0, pin_memory=True, collate_fn=collate_pad_train)

dl_va = DataLoader(ds_va, batch_size=1, shuffle=False,
                   num_workers=0, pin_memory=True, collate_fn=collate_pad_train)

dl_te = DataLoader(
    ds_te,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
    collate_fn=collate_pad_test,      # <-- new
)

len(train_imgs), len(tr_imgs), len(val_imgs), len(dl_te), len(dl_va)

(1900, 1710, 190, 250, 190)

# MODELLING (REGRESSION)

In [30]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):                      # x: [B,C,H,W]
        x = x.clamp(min=self.eps)
        x = x**self.p
        x = F.avg_pool2d(x, (x.size(-2), x.size(-1)))
        return x**(1.0/self.p)

def build_model(name="resnet50", dropout=0.25, gem_p=3.0):
    assert name == "resnet50", "this head is wired for resnet50"
    backbone = tv.models.resnet50(weights=tv.models.ResNet50_Weights.IMAGENET1K_V2)
    # replace avgpool -> GeM
    backbone.avgpool = GeM(p=gem_p)
    in_feats = backbone.fc.in_features
    # a slightly deeper regressor; small and stable
    backbone.fc = nn.Sequential(
        nn.Linear(in_feats, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(512, 128),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(128, 1),
    )
    return backbone

In [31]:
# --- Build model (uses your GeM head) ---
model = build_model("resnet50", dropout=0.25).to(device)
model = model.to(memory_format=torch.channels_last)

# --- Log-target helpers (for stability across large count ranges) ---
LOG_TARGETS = True  # set False to go back to plain counts
LOG_CAP = 15.0      # guard to prevent exp overflow during inverse transform

def to_target(y):
    return torch.log1p(y) if LOG_TARGETS else y

def from_pred(yhat):
    if LOG_TARGETS:
        yhat = torch.clamp(yhat, max=LOG_CAP)
        return torch.expm1(yhat)
    return yhat

# --- Freeze early layers BEFORE creating the optimizer ---
for p in list(model.parameters())[:-4]:
    p.requires_grad = False

# --- Steps accounting (must know this BEFORE defining the warmup fn) ---
steps_per_epoch = max(1, math.ceil(len(ds_tr) / CFG["batch_size"]))
num_steps = CFG["epochs"] * steps_per_epoch
WARMUP_STEPS = max(50, num_steps // 20)  # ~5% of total steps or at least 50

def lr_warmup_cos(step: int):
    """
    Linear warmup for WARMUP_STEPS steps, then cosine decay to 0.
    'step' is the GLOBAL optimizer step count (0-based).
    """
    if step < WARMUP_STEPS:
        return float(step + 1) / float(WARMUP_STEPS)
    t = (step - WARMUP_STEPS) / max(1, (num_steps - WARMUP_STEPS))
    return 0.5 * (1.0 + math.cos(math.pi * t))

# --- Optimizer / Scheduler / AMP scaler ---
# Tip: keep LR modest for AdamW on this setup, e.g. 3e-4 or 5e-4
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["lr"], weight_decay=CFG["weight_decay"]
)

# IMPORTANT: pass the FUNCTION, not lr_warmup_cos(num_steps)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_warmup_cos)

scaler = torch.cuda.amp.GradScaler(enabled=CFG["amp"])

print(f"steps_per_epoch={steps_per_epoch}, num_steps={num_steps}, warmup={WARMUP_STEPS}")


steps_per_epoch=855, num_steps=12825, warmup=641


/tmp/ipykernel_55754/1600022705.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=CFG["amp"])


# TRAINING AND VALIDATE LOOPS

In [32]:
criterion = nn.SmoothL1Loss(beta=2.0)
mae = nn.L1Loss(reduction="none")

def run_epoch(model, loader, train=True):
    model.train(train)
    total_loss, total_abs_err, n = 0.0, 0.0, 0
    for imgs, targets in loader:
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        tgt = to_target(targets)  # <-- transform target

        with torch.amp.autocast("cuda", enabled=CFG["amp"]), torch.set_grad_enabled(train):
            preds = model(imgs)
            loss = criterion(preds, tgt)

        if train:
            scaler.scale(loss).backward()
            # unscale then clip
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

        # MAE in the original count space
        cnt_pred = from_pred(preds.detach())
        abs_err = mae(cnt_pred, targets).sum().item()

        total_abs_err += abs_err
        total_loss += loss.detach().item() * imgs.size(0)
        n += imgs.size(0)

        del imgs, targets, preds, loss
    return total_loss/max(n,1), total_abs_err/max(n,1)

# Taining schedule: warmup

In [33]:
# quick warmup for first few hundred steps
WARMUP_STEPS = max(50, num_steps // 20)

def lr_warmup_cos(step):
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    # cosine on the remaining steps
    t = (step - WARMUP_STEPS) / max(1, (num_steps - WARMUP_STEPS))
    return 0.5 * (1 + math.cos(math.pi * t))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_warmup_cos)

# progressive unfreeze (optional on 4GB VRAM):
#  - first 2-3 epochs train only the head
#  - then unfreeze all and keep training
for p in list(model.parameters())[:-4]:
    p.requires_grad = False

UNFREEZE_AT_EPOCH = 3


In [34]:
best_val_mae = float("inf")
os.makedirs("checkpoints", exist_ok=True)
for epoch in range(1, CFG["epochs"]+1):
    if epoch == UNFREEZE_AT_EPOCH:
        for p in model.parameters():
            p.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_warmup_cos)
        print(">> Unfroze all layers and reset optimizer.")

    def set_bn_eval(m):
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

    if epoch < UNFREEZE_AT_EPOCH:
        model.apply(set_bn_eval)

    t0 = time.time()
    tr_loss, tr_mae = run_epoch(model, dl_tr, train=True)
    va_loss, va_mae = run_epoch(model, dl_va, train=False)

    if va_mae < best_val_mae:
        best_val_mae = va_mae
        torch.save({"epoch": epoch, "model": model.state_dict(), "cfg": CFG}, "checkpoints/best.pth")

    print(f"Epoch {epoch:02d} | "
          f"train loss {tr_loss:.4f} mae {tr_mae:.3f} | "
          f"val loss {va_loss:.4f} mae {va_mae:.3f} | "
          f"lr {optimizer.param_groups[0]['lr']:.2e} | "
          f"{time.time()-t0:.1f}s")

print("Best val MAE:", best_val_mae)

Epoch 01 | train loss 2.8232 mae 138.140 | val loss 3.0034 mae 146.873 | lr 7.80e-06 | 59.3s
Epoch 02 | train loss 2.8209 mae 138.138 | val loss 3.0160 mae 146.885 | lr 7.80e-06 | 60.2s
>> Unfroze all layers and reset optimizer.
Epoch 03 | train loss 2.8205 mae 138.137 | val loss 3.0106 mae 146.880 | lr 7.80e-06 | 114.9s
Epoch 04 | train loss 2.8185 mae 138.136 | val loss 3.0067 mae 146.876 | lr 7.80e-06 | 113.4s
Epoch 05 | train loss 2.8273 mae 138.143 | val loss 3.0162 mae 146.885 | lr 7.80e-06 | 112.4s
Epoch 06 | train loss 2.8229 mae 138.139 | val loss 3.0059 mae 146.875 | lr 7.80e-06 | 113.8s
Epoch 07 | train loss 2.8209 mae 138.138 | val loss 2.9992 mae 146.869 | lr 7.80e-06 | 113.6s
Epoch 08 | train loss 2.8186 mae 138.136 | val loss 3.0153 mae 146.884 | lr 7.80e-06 | 113.4s
Epoch 09 | train loss 2.8165 mae 138.134 | val loss 3.0131 mae 146.882 | lr 7.80e-06 | 113.4s
Epoch 10 | train loss 2.8223 mae 138.140 | val loss 3.0144 mae 146.883 | lr 7.80e-06 | 113.1s
Epoch 11 | train lo

# INSPECT

In [44]:
with torch.inference_mode():
    imgs, ys = next(iter(dl_va))
    preds = model(imgs.to(device)).cpu().squeeze(1)
for i in range(min(5, len(imgs))):
    print(f"GT={ys[i].item():.1f} | Pred={preds[i].item():.1f}")


GT=58.0 | Pred=59.8
GT=14.0 | Pred=9.6


# INFERENCE TO TEST SET

In [45]:
def _pad_to(img, H, W):
    _, h, w = img.shape
    return F.pad(img, (0, W - w, 0, H - h), value=0)

def collate_pad_test(batch):
    # batch: list of (tensor, filename_str)
    imgs, names = zip(*batch)
    H = max(t.shape[1] for t in imgs)
    W = max(t.shape[2] for t in imgs)
    imgs = torch.stack([_pad_to(t, H, W) for t in imgs], dim=0).to(memory_format=torch.channels_last)
    return imgs, list(names)

In [46]:
# ==== Ultra-safe test-time inference for 4 GB GPUs ====

# 0) Build test dataset
test_imgs = sorted([Path(p) for p in glob.glob(str(TEST_IMG_DIR / "*"))
                    if p.lower().endswith((".jpg",".jpeg",".png"))])
ds_te = CrowdCountDataset(test_imgs, lbl_dir=None, train=False)

# 1) Use batch_size=1 to avoid huge per-batch padding
#    -> no custom collate needed at all
dl_te = DataLoader(
    ds_te,
    batch_size=1,
    shuffle=False,
    num_workers=0,         # lower memory pressure
    pin_memory=True,
)

SCALES = [448, 512, 576]  # test-only short-sides
USE_FLIP = True

def tta_predict(img_pil: Image.Image) -> float:
    preds = []
    for s in SCALES:
        # rebuild a minimal transform for test-only resizing/norm
        t = transforms.Compose([
            transforms.Lambda(lambda im: tv.transforms.functional.resize(im, size=s, antialias=True)),
            LetterboxPad(CFG["pad_to"]),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ])
        x = t(img_pil).unsqueeze(0).to(device).to(memory_format=torch.channels_last)
        with torch.inference_mode(), torch.amp.autocast("cuda", enabled=CFG["amp"]):
            p = model(x)
            preds.append(p)
            if USE_FLIP:
                xf = torch.flip(x, dims=[3])
                pf = model(xf)
                preds.append(pf)
        del x
    yhat = torch.mean(torch.cat(preds, dim=0), dim=0, keepdim=False)  # [1]
    return float(torch.clamp(from_pred(yhat), min=0.0).item())

# Build test set without a DataLoader (loads 1-by-1 to save memory)
test_imgs = sorted([Path(p) for p in glob.glob(str(TEST_IMG_DIR / "*"))
                    if p.lower().endswith((".jpg",".jpeg",".png"))])

# 2) Load checkpoint safely and prep model
ckpt = torch.load("checkpoints/best.pth", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()
model = model.to(memory_format=torch.channels_last)

# 3) Inference loop with AMP + OOM fallback to CPU per image
pred_rows = []
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# prefer fp16 on CUDA for memory; if your GPU prefers bf16 you can use dtype=torch.bfloat16
amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16

with torch.inference_mode(), torch.amp.autocast("cuda", enabled=(device.type=="cuda"), dtype=amp_dtype):
    for imgs, names in dl_te:
        try:
            # imgs: [1, C, H, W]
            imgs = imgs.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            preds = model(imgs).detach().cpu().squeeze(1).numpy()
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and device.type == "cuda":
                # Per-image fallback to CPU if even bs=1 OOMs (rare but possible for very large frames)
                print(f"[WARN] OOM on {names[0]} — retrying on CPU.")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                imgs_cpu = imgs.cpu()
                with torch.inference_mode():
                    preds = model.to("cpu")(imgs_cpu).detach().squeeze(1).numpy()
                model.to(device)  # move back
            else:
                raise

        preds = np.clip(preds, 0, None)
        pred_rows.append((names[0], float(preds[0])))

len(pred_rows), pred_rows[:3]


(500, [('1.jpg', 17.015625), ('10.jpg', 99.4375), ('100.jpg', 105.9375)])

# SUBMIT CSV

In [47]:
# Try to infer expected column names from sample submission.
# Commonly: ["image_id", "count"] or ["Id","Predicted"] etc.
if SUB_TEMPLATE.exists():
    sub_template = pd.read_csv(SUB_TEMPLATE)
    cols = list(sub_template.columns)
    print("Sample submission columns:", cols)
    # Guess which column holds the filename key
    key_col = [c for c in cols if "id" in c.lower() or "image" in c.lower()][0]
    pred_col = [c for c in cols if c != key_col][0]
    # Map filename -> predicted
    df_pred = pd.DataFrame(pred_rows, columns=["filename","pred"])
    # Try to merge on filename; if template stores plain names or with extension differences, normalize both.
    T = sub_template.copy()
    T["_key"] = T[key_col].astype(str).str.replace(r"^\.?/","", regex=True)
    df_pred["_key"] = df_pred["filename"].astype(str)
    out = T.merge(df_pred[["_key","pred"]], on="_key", how="left").drop(columns="_key")
    out[pred_col] = out["pred"].fillna(0).round(3)
    out = out[[key_col, pred_col]]
else:
    # Fallback if no template: assume ["image_id","count"]
    out = pd.DataFrame(pred_rows, columns=["image_id","count"])
    out["count"] = out["count"].round(3)

SAVE_PATH = "../Submission/sub_resnet50_no-color-jitter_lr001.csv"
out.to_csv(SAVE_PATH, index=False)
print("Wrote:", SAVE_PATH)
out.head()


Sample submission columns: ['image_id', 'predicted_count']
Wrote: ../Submission/sub_resnet50_no-color-jitter_lr001.csv


,image_id,predicted_count
0,1.jpg,17.016
1,2.jpg,86.250
2,3.jpg,123.062
3,4.jpg,54.812
4,5.jpg,85.125
